# NB11 — Leave-One-Family-Out (LOFO) Cross-Family Zero-Day Experiment

**Goal (HANDOFF §14 item 12):** Test cross-family zero-day detection. Current within-family
results (NB07b/NB09) hold out Conti *variants* but Conti base IS in training — so those
are within-family results. LOFO asks: can the detector flag a ransomware *family* it has
never seen during training at all?

**Setup:** 6 folds over the 6 ransomware families (WannaCry, Ryuk, REvil, LockBit, Darkside, Conti).
Each fold:
- Hold out 1 entire ransomware family (10 executions) as the test set
- Train on: the other 5 families + all legit-encryption + all benign (110 executions)
- Measure: detection rate on the 10 held-out execs (% called Ransomware)
- Also check: val legit-enc and benign recall (confirm no FPR regression per fold)

**Metrics:**
- Per-fold detection rate (10-exec test set per fold)
- Mean cross-family zero-day recall across 6 folds
- Compare to within-family: static 89% (NB07b), streaming 97% (NB09)

**Architecture:** same TextEmbeddingClassifier as NB07b/NB10 — retrained from scratch per fold
so each fold's model has never seen the held-out family.

**Conti variants (Conti_01–07) are ALWAYS excluded** from LOFO — they are a separate
within-family test set and must not contaminate cross-family training or testing.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import json
import time

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
FEATURES_DIR = 'features'

# Classifier hyperparams — same as NB07b for fair comparison
EMBED_DIM  = 384
N_WINDOWS  = 300
N_CLASSES  = 3
EPOCHS     = 80
LR         = 1e-3

CLASS_ORDER = ['Benign', 'Legit-Encryption', 'Ransomware']

LABEL_3WAY = {
    'Benign':           ['Firefox', 'Office', 'Idle'],
    'Legit-Encryption': ['AESCrypt', 'SDelete', 'Zip'],
    'Ransomware':       ['WannaCry', 'Ryuk', 'REvil', 'LockBit', 'Darkside', 'Conti'],
}
RANSOMWARE_FAMILIES = ['WannaCry', 'Ryuk', 'REvil', 'LockBit', 'Darkside', 'Conti']
CONTI_VARIANTS      = [f'Conti_0{i}' for i in range(1, 8)]

# Streaming params (NB09 settings — used if streaming=True)
STREAM_W      = 50
STREAM_STRIDE = 5
STREAM_THRESH = 0.70
RANSOMWARE_IDX = CLASS_ORDER.index('Ransomware')

print(f'Device: {DEVICE}')
print(f'Families for LOFO: {RANSOMWARE_FAMILIES}')
print(f'Conti variants (always excluded): {CONTI_VARIANTS}')

In [ ]:
# Load embeddings + metadata
X_text = np.load(f'{FEATURES_DIR}/text_sequences.npy')   # (190, 300, 384)
meta   = pd.read_csv(f'{FEATURES_DIR}/metadata.csv')

# Build 3-way label column
label_map = {m: c for c, members in LABEL_3WAY.items() for m in members}
meta['label_3way'] = meta['label'].map(label_map)

# Separate originals from Conti variants
# Variants are ALWAYS excluded — separate zero-day test (within-family)
orig_mask = meta['group'] == 'original'
orig_idx  = np.where(orig_mask)[0]   # 120 originals — indices into X_text / meta
var_idx   = np.where(~orig_mask)[0]  # 70 Conti variants

print(f'Original executions : {len(orig_idx)}')
print(f'Conti variant execs : {len(var_idx)}  (excluded from all LOFO folds)')
print(f'X_text shape        : {X_text.shape}')
print()
print('Original label counts:')
print(meta.iloc[orig_idx]['label'].value_counts().sort_index().to_string())

# Sanity check: all Conti variants excluded
assert all(meta.iloc[var_idx]['label'].isin(CONTI_VARIANTS)), \
    'variant indices contain non-variant labels!'
print(f'\nSanity check passed — {len(var_idx)} Conti variant execs correctly identified.')

In [ ]:
# Model definition — identical to NB07b / NB10 TextEmbeddingClassifier
# Retrained from scratch per LOFO fold (fresh random init each fold)
class TextEmbeddingClassifier(nn.Module):
    def __init__(self, seq_len=N_WINDOWS, input_dim=EMBED_DIM, num_classes=N_CLASSES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # global mean pool — sequence-length agnostic
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )
        self.reconstructor = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, seq_len * input_dim),
        )

    def forward(self, x):
        # x: (B, seq_len, input_dim)
        pooled = self.conv(x.permute(0, 2, 1)).squeeze(-1)   # (B, 64)
        return self.classifier(pooled), self.reconstructor(pooled), pooled


def train_fold(X_train_np, y_train_np, X_val_np, y_val_np,
               epochs=EPOCHS, lr=LR, seed=42):
    """Train a fresh classifier for one LOFO fold. Returns (model, val_acc)."""
    torch.manual_seed(seed)
    model = TextEmbeddingClassifier().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()

    Xt = torch.FloatTensor(X_train_np).to(DEVICE)
    yt = torch.LongTensor(y_train_np).to(DEVICE)
    Xv = torch.FloatTensor(X_val_np).to(DEVICE)
    yv = torch.LongTensor(y_val_np).to(DEVICE)

    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        logits, _, _ = model(Xt)
        crit(logits, yt).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_acc = (model(Xv)[0].argmax(1) == yv).float().mean().item()

    return model, val_acc


@torch.no_grad()
def streaming_detect(model, embedding_np,
                     W=STREAM_W, stride=STREAM_STRIDE, thresh=STREAM_THRESH):
    """Return True if streaming max P(Ransomware) >= thresh for this execution."""
    n = embedding_np.shape[0]
    max_p = 0.0
    for start in range(0, n - W + 1, stride):
        x = torch.FloatTensor(embedding_np[start:start + W]).unsqueeze(0).to(DEVICE)
        logits, _, _ = model(x)
        p = torch.softmax(logits, 1)[0, RANSOMWARE_IDX].item()
        if p > max_p:
            max_p = p
    return max_p >= thresh, max_p


print('TextEmbeddingClassifier defined.')
print('train_fold() and streaming_detect() ready.')

In [ ]:
# LOFO experiment — 6 folds
# Each fold: hold out 1 ransomware family, train on the other 5 + legit-enc + benign

le = LabelEncoder()
le.fit(CLASS_ORDER)
ran_label  = le.transform(['Ransomware'])[0]
lenc_label = le.transform(['Legit-Encryption'])[0]
ben_label  = le.transform(['Benign'])[0]

RUN_STREAMING = True   # set False to skip streaming (faster)

fold_results = []

print(f'{'═'*80}')
print(f'  LOFO — Cross-Family Zero-Day Experiment  ({EPOCHS} epochs / fold)')
print(f'{'═'*80}')
hdr = f'{"Held-out":>10} | {"N-train":>7} | {"ValAcc":>6} | {"LEnc-R":>6} | {"Ben-R":>5} | {"Static":>8} | {"Stream":>8}'
print(hdr)
print(f'{'─'*80}')

for fam in RANSOMWARE_FAMILIES:
    t0 = time.time()

    # ── Indices ──────────────────────────────────────────────────────────────
    orig_labels = meta.iloc[orig_idx]['label'].values

    pool_mask = orig_labels != fam
    test_mask = orig_labels == fam

    pool_abs = orig_idx[pool_mask]   # global indices for training pool
    test_abs = orig_idx[test_mask]   # global indices for test (held-out family)

    # ── Pool split (stratified 80/20) ─────────────────────────────────────
    y_pool = le.transform(meta.iloc[pool_abs]['label_3way'].values)
    local_tr, local_va = train_test_split(
        np.arange(len(pool_abs)), test_size=0.2, stratify=y_pool, random_state=42
    )

    X_train = X_text[pool_abs[local_tr]]
    y_train = y_pool[local_tr]
    X_val   = X_text[pool_abs[local_va]]
    y_val   = y_pool[local_va]
    X_test  = X_text[test_abs]          # held-out family

    # ── Train ─────────────────────────────────────────────────────────────
    model, val_acc = train_fold(X_train, y_train, X_val, y_val)

    # ── Static evaluation on held-out family ──────────────────────────────
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(torch.FloatTensor(X_test).to(DEVICE))
        preds_test   = logits.argmax(1).cpu().numpy()

    static_n    = int((preds_test == ran_label).sum())
    static_rate = static_n / len(X_test)

    # ── Val per-class recall ─────────────────────────────────────────────
    with torch.no_grad():
        val_preds = model(torch.FloatTensor(X_val).to(DEVICE))[0].argmax(1).cpu().numpy()

    val_lenc_r = ((val_preds == lenc_label) & (y_val == lenc_label)).sum() / max(1, (y_val == lenc_label).sum())
    val_ben_r  = ((val_preds == ben_label)  & (y_val == ben_label)).sum()  / max(1, (y_val == ben_label).sum())

    # ── Streaming evaluation on held-out family ───────────────────────────
    if RUN_STREAMING:
        stream_hits = 0
        for idx in test_abs:
            fired, _ = streaming_detect(model, X_text[idx])
            if fired:
                stream_hits += 1
        stream_n    = stream_hits
        stream_rate = stream_n / len(test_abs)
        stream_str  = f'{stream_n}/10 ({stream_rate*100:.0f}%)'
    else:
        stream_rate = float('nan')
        stream_str  = 'skipped'

    elapsed = time.time() - t0
    print(
        f'{fam:>10} | {len(X_train):>7} | {val_acc:6.3f} | {val_lenc_r:6.3f} | {val_ben_r:5.3f} |'
        f' {static_n}/10 ({static_rate*100:.0f}%) | {stream_str}  [{elapsed:.0f}s]'
    )

    fold_results.append({
        'family':       fam,
        'n_train':      int(len(X_train)),
        'n_val':        int(len(X_val)),
        'n_test':       int(len(X_test)),
        'val_acc':      float(val_acc),
        'val_lenc_r':   float(val_lenc_r),
        'val_ben_r':    float(val_ben_r),
        'static_n':     static_n,
        'static_rate':  float(static_rate),
        'stream_n':     int(stream_n) if RUN_STREAMING else None,
        'stream_rate':  float(stream_rate) if RUN_STREAMING else None,
    })

print(f'{'─'*80}')
mean_static  = np.mean([r['static_rate']  for r in fold_results])
mean_stream  = np.mean([r['stream_rate']  for r in fold_results if r['stream_rate'] is not None])
print(f'{"MEAN":>10} | {"":>7} | {"":>6} | {"":>6} | {"":>5} |'
      f' {mean_static*100:5.1f}%     | {mean_stream*100:.1f}%')
print()
print(f'Cross-family zero-day recall (static)  : {mean_static*100:.1f}%')
print(f'Cross-family zero-day recall (streaming): {mean_stream*100:.1f}%')
print(f'Within-family (static NB07b)           : 88.6%')
print(f'Within-family (streaming NB09)         : 97.1%')

In [ ]:
# Per-fold summary table (clean format for Results.md)
print(f'{'═'*72}')
print(f'  Per-Fold Detection Rate — Cross-Family Zero-Day')
print(f'{'═'*72}')
print(f'{"Family":>12} | {"Static detect":>13} | {"Streaming detect":>16} | {"Val acc":>7}')
print(f'{'─'*72}')
for r in fold_results:
    s_str   = f"{r['static_n']}/10 ({r['static_rate']*100:.0f}%)"
    st_str  = f"{r['stream_n']}/10 ({r['stream_rate']*100:.0f}%)" if r['stream_rate'] is not None else 'N/A'
    print(f"{r['family']:>12} | {s_str:>13} | {st_str:>16} | {r['val_acc']:>7.3f}")
print(f'{'─'*72}')
mean_static = np.mean([r['static_rate']  for r in fold_results])
mean_stream = np.mean([r['stream_rate']  for r in fold_results if r['stream_rate'] is not None])
print(f'{"MEAN":>12} | {mean_static*100:10.1f}%   | {mean_stream*100:13.1f}%    |')
print()
print('Reference — within-family (NB07b static: 88.6% | NB09 streaming: 97.1%)')
print()
print('Val FPR summary (mean across folds):')
mean_lenc_r = np.mean([r['val_lenc_r'] for r in fold_results])
mean_ben_r  = np.mean([r['val_ben_r']  for r in fold_results])
print(f'  Legit-Enc recall (val, mean): {mean_lenc_r*100:.1f}%  (should stay high — confirms no Legit-Enc regression)')
print(f'  Benign recall (val, mean)   : {mean_ben_r*100:.1f}%  (should stay high — confirms no Benign regression)')

In [ ]:
# Save results
import json as _json

lofo_output = {
    'config': {
        'epochs': EPOCHS,
        'lr': LR,
        'architecture': 'TextEmbeddingClassifier (same as NB07b/NB10)',
        'stream_W': STREAM_W,
        'stream_stride': STREAM_STRIDE,
        'stream_thresh': STREAM_THRESH,
        'val_split': 0.2,
        'random_state': 42,
        'note': 'Conti_01–07 excluded from all folds (within-family zero-day set)',
    },
    'folds': fold_results,
    'mean_static_recall':  float(mean_static),
    'mean_stream_recall':  float(mean_stream) if RUN_STREAMING else None,
    'within_family_static_ref':    0.886,
    'within_family_streaming_ref': 0.971,
}

out_path = f'{FEATURES_DIR}/lofo_results.json'
with open(out_path, 'w') as f:
    _json.dump(lofo_output, f, indent=2)
print(f'Saved → {out_path}')